In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
import pandas as pd
from limb_fitting import *
from utils import *
from fitting import *
from interpolation import *
from reprojection import *

In [51]:
def show_data(image, to_file=None, **kwargs):
    if to_file is not None:
        plt.ioff()

    fig = plt.figure(figsize=(10,10))
    plt.imshow(image, **kwargs)
    plt.tight_layout()

    if to_file is not None:
        plt.savefig(to_file)
        plt.ion()
        plt.close(fig)


def get_scale(img_data):
    if img_data is not None:
        fmt, rng = img_data['PHI_IMG_format'], img_data['PHI_IMG_maxRange']

        scale = rng[-1] / rng[0]
        if fmt[-1] != 'IMGFMT_24_8':
            scale *= 256
        return scale
    else:
        return None


def CLD(mu, approximation='Neckel'):
    if approximation == 'Neckel':
        p = [0.48767921486914473,
             -1.6848471461910317,
             2.355950448408068,
             -1.827014432405401,
             1.3540877312885482,
             0.31414418403067174]
    else:
        a, b = -0.1, -0.45
        p = [a, -2 * a - b, a + b + 1]
    return np.polyval(p, mu) * (mu > 0)

In [3]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers_cor.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz')
xu, yu = s['xu'], s['yu']
xd, yd = s['xd'], s['yd']

In [52]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20260126T223810_V202602200142C_0621261001.fits.gz'

with fits.open(dark_file) as hdul:
    dark_header = hdul[0].header
    dark = hdul[0].data

1.0


In [5]:
flat_file = '/home/ulyanov/data/solo/phi/flat/fdt/transmittance/phi-fdt-flat_20250310T080009_V202511271432C_0563100100.fits'

with fits.open(flat_file) as hdul:
    flat_header = hdul[0].header
    flat = hdul[0].data

In [6]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flare trigger/2026-02-04/*'))

In [128]:
x0 = 600
y0 = 600
h0 = 64

xq, yq = 614, 820
#xq, yq = 690, 850
hq = 80

scale0 = None

Q = []
X = []
Y = []

for j, file in enumerate(files[:]):
    with fits.open(file) as hdul:
        header = hdul[0].header
        data = hdul[0].data
        img_data = hdul['PHI_FITS_imageSummary'].data.copy()
        scale = get_scale(img_data)

    data = (data - scale * crop(dark, header)) / crop(flat[-1], header)

    for i in range(len(data)):
        temp = data[i].copy()

        for _ in range(1):
            xc, yc, r_sun = find_center(temp)
            dx, dy = x0 - xc, y0 - yc

            xi, yi = np.mgrid[:temp.shape[0], :temp.shape[1]]
            mu = np.sqrt(1 - ((xi - xc) ** 2 + (yi - yc) ** 2) / r_sun ** 2)

            q = polyfit2d(temp / CLD(mu), xi, yi, degree=1)#, niter=10, sigma=1000)
            temp /= q

        xc, yc, r_sun = find_center(temp)
        dx, dy = x0 - xc, y0 - yc

        X += [dx]
        Y += [dy]

        data[i] = interp2d(temp, dx, dy, roll=True, kind='quadratic')


    Q.append(list(data[:,xq-hq:xq+hq, yq-hq:yq+hq]))

Q = np.array(Q).reshape((-1, 2 * hq, 2 * hq))

X = np.array(X)
Y = np.array(Y)

X -= np.mean(X)
Y -= np.mean(Y)

X /= np.std(X)
Y /= np.std(Y)

/tmp/ipykernel_102001/2656540795.py:32: RuntimeWarning: invalid value encountered in sqrt
  mu = np.sqrt(1 - ((xi - xc) ** 2 + (yi - yc) ** 2) / r_sun ** 2)


In [129]:
delta = 10

for i in range(len(Q)):
    Q0 = Q[max(i - delta,0)]
    #show_data(Q[i] - Q0, cmap='gray', vmin=-0.02, vmax=0.02, to_file=f'temp/{i:03d}.png')
    show_data(Q[i], cmap='inferno', vmin=0.6, vmax=1.0, to_file=f'temp/{i:03d}.png')

In [127]:
x1, y1 = 104, 101
x2, y2 = 111, 108

h = 0

plt.figure(figsize=(10,8))
plt.plot(np.mean(Q[:,x1-h:x1+h+1,y1-h:y1+h+1], axis=(1,2)))
plt.plot(np.mean(Q[:,x2-h:x2+h+1,y2-h:y2+h+1], axis=(1,2)))

In [72]:
x1, y1 = 104, 101

q = Q[:,x1-1:x1+2,y1-1:y1+2]

delta = 10
q = q[delta:] - q[:-delta]


fig, ax = plt.subplots(figsize=(10,8))
ax_ = ax.twinx()

#ax.plot(q.reshape((len(q),-1)))
#ax.plot(q[:,1,2])
ax.plot(q[:,0,2])
ax_.plot(X[delta:] - X[:-delta], color='red')
#ax_.plot(Y - np.mean(Y), color='red')

In [94]:
temp = Q[:,x1-1, y1+1].copy()

plt.figure(figsize=(10,8))
plt.plot(temp)

In [90]:
np.mean(temp * X)

np.float64(0.0009224768840651857)

In [133]:
4 / 24 / 60 / 25 * 2 * np.pi * 500

0.3490658503988659

In [132]:
r_sun

np.float64(461.20434059594254)